In [ ]:
# 1. 외부 모듈 자동 새로고침 설정 (loader.py 수정 시 즉각 반영)
%load_ext autoreload
%autoreload 2

# 2. 필수 라이브러리 임포트
import os
import json
import pandas as pd
import FinanceDataReader as fdr
import pykrx
import OpenDartReader
import matplotlib
import seaborn
import scipy
from datetime import date

# 3. 직접 만든 로컬 모듈 임포트
from data.loader import QuantDataLoader

print("✅ 환경 설정 및 전체 라이브러리 정상 로드 완료!")

In [ ]:
import pandas as pd
from datetime import date

# 구현해두신 모듈들을 임포트합니다.
# (경로나 클래스명은 실제 프로젝트 환경에 맞게 조정해 주세요)
from data.loader import QuantDataLoader
from stages.stage1_neglected_sector import NeglectedSectorScreener

print("==================================================")
print("🚀 [실전 테스트] Stage 1: KOSPI 소외 섹터 발굴")
print("==================================================\n")

# 1. 기준일 설정 (현재 날짜 기준)
base_date = date(2026, 7, 31)

# 2. 파라미터 세팅 (config/params.yaml에서 불러오는 것을 모사)
stage1_params = {
    'stage1_return_weight': 0.5,
    'stage1_volume_weight': 0.5,
    'stage1_pass_ratio': 0.4  # 상위 40% 섹터 통과
}

try:
    # 3. 로더 초기화 및 실제 섹터 데이터 조달
    # (내부적으로 pykrx 등을 호출하여 데이터를 가져온다고 가정합니다)
    loader = QuantDataLoader()
    
    print("⏳ KOSPI 섹터 시계열 데이터 수집 중...")
    sector_metrics_df = loader.get_sector_metrics(base_date)
    
    if sector_metrics_df.empty:
        print("❌ 섹터 데이터를 불러오지 못했습니다. 로더의 구현 상태를 확인해 주세요.")
    else:
        # 4. Stage 1 스크리너 실행
        screener = NeglectedSectorScreener(params=stage1_params)
        passed_sectors = screener.run(sector_metrics_df)
        
        print(f"\n📊 [기준일: {base_date}] KOSPI 전체 섹터 데이터 (상위 5개)")
        display(sector_metrics_df.head())
        
        print(f"\n✅ [Stage 1 통과] 최종 소외 섹터 (Pass Ratio: {stage1_params['stage1_pass_ratio']*100}%)")
        display(passed_sectors)
        
        # 밸류트랩 경고 확인
        value_traps = passed_sectors[passed_sectors['is_value_trap_warning'] == True]
        if not value_traps.empty:
            print("\n⚠️ [주의] 다음 섹터는 통과되었으나 낙폭 가속(Value Trap) 위험이 감지되었습니다:")
            print(value_traps['sector'].tolist())

except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from datetime import date

from data.loader import QuantDataLoader
from stages.stage2_sector_leaders import SectorLeaderScreener

print("==================================================")
print("🚀 [실전 테스트] Stage 2: 섹터 내 우량주 탐색")
print("==================================================\n")

# 1. 기준일 설정 (현재 날짜 기준)
base_date = date(2026, 7, 31)

# 2. Stage 2 파라미터 세팅
stage2_params = {
    'roe_percentile_cutoff': 0.5,
    'roic_percentile_cutoff': 0.5,
    'op_margin_std_percentile_cutoff': 0.5,
    'op_margin_lookback_q': 8,
    'op_margin_min_quarters': 4
}

try:
    loader = QuantDataLoader()
    screener = SectorLeaderScreener(params=stage2_params)
    
    # 3. 테스트용 실제 종목 리스트 
    # (Stage 1을 통과한 종목들이라고 가정하고, 익숙한 KOSPI 종목들로 구성)
    test_tickers_df = pd.DataFrame({
        'ticker': ['005930', '000660', '005380', '068270'],  # 삼성전자, SK하이닉스, 현대차, 셀트리온
        'sector': ['IT', 'IT', '자동차', '바이오'] 
    })
    
    print("⏳ DART API 실전 재무 데이터 수집 및 지표 계산 중...")
    print("(종목별로 과거 8개 분기 원자료를 가져와야 하므로 시간이 약간 소요될 수 있습니다.)\n")
    
    # 4. Stage 2 스크리너 실행
    passed_stage2_df = screener.run(test_tickers_df, loader, base_date)
    
    print("📊 [입력된 테스트 종목]")
    display(test_tickers_df)
    
    print("\n✅ [Stage 2 통과] 최종 섹터 우량주")
    display(passed_stage2_df)
    
except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from datetime import date

from data.loader import QuantDataLoader
from stages.stage3_fundamental_improve import FundamentalImproveScreener

print("==================================================")
print("🚀 [실전 테스트] Stage 3: 체질 개선 (Turnaround)")
print("==================================================\n")

# 1. 기준일 설정 (현재 날짜 기준)
base_date = date(2026, 7, 31)

# 2. Stage 3 파라미터 세팅
stage3_params = {
    'sga_lookback_quarters': 6,
    'sales_weight': 0.4,
    'sga_weight': 0.4,
    'gpm_weight': 0.2,
    'composite_pass_percentile': 0.5, # 종목 수 확보를 위해 0.5로 완화
    'cost_cutting_only_sales_decline_threshold': -0.05
}

try:
    loader = QuantDataLoader()
    screener = FundamentalImproveScreener(params=stage3_params)
    
    # 3. 테스트용 실제 종목 리스트 
    # (Stage 2를 통과했다고 가정. 예외 처리 작동 확인을 위해 금융주 추가)
    test_tickers_df = pd.DataFrame({
        'ticker': ['005930', '000660', '005380', '105560'],  # 삼성전자, SK하이닉스, 현대차, KB금융
        'sector': ['IT', 'IT', '자동차', '금융'] 
    })
    
    print("⏳ DART API 실전 재무 데이터(최근 6개 분기 시계열) 수집 중...")
    print("(각 종목별로 과거 6분기치 데이터를 조합해야 하므로 시간이 소요될 수 있습니다.)\n")
    
    # 4. Stage 3 스크리너 실행
    passed_stage3_df = screener.run(test_tickers_df, loader, base_date)
    
    print("📊 [입력된 테스트 종목]")
    display(test_tickers_df)
    
    print("\n✅ [Stage 3 통과] 펀더멘털 개선(Turnaround) 확인 종목")
    display(passed_stage3_df)
    
except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from datetime import date
from data.loader import QuantDataLoader
from stages.stage4_valuation import ValuationScreener

print("==================================================")
print("🚀 [실전 테스트] Stage 4: 밸류에이션 (Valuation)")
print("==================================================\n")

# 1. 기준일 설정
base_date = date(2026, 7, 31)

# 2. 파라미터 설정
stage4_params = {
    'pbr_weight': 0.5,
    'bps_growth_weight': 0.5,
    'composite_pass_percentile': 0.5, # 종목 수 확보를 위해 0.5로 완화
    'value_trap_roe_threshold': 0.05
}

try:
    loader = QuantDataLoader()
    screener = ValuationScreener(params=stage4_params)
    
    # 3. 테스트용 종목 리스트 (Stage 3 통과 가정)
    # 4단계는 밸류트랩 검증을 위해 Stage 2에서 계산된 'roe' 값이 필수로 필요하므로 가상의 ROE를 주입합니다.
    test_tickers_df = pd.DataFrame({
        'ticker': ['005930', '000660', '015760', '005380'],  # 삼성전자, SK하이닉스, 한국전력, 현대차
        'sector': ['IT', 'IT', '유틸리티', '자동차'],
        'roe': [0.12, 0.15, 0.01, 0.10]  # 한국전력(015760)을 밸류트랩으로 가정하여 ROE 1% 부여
    })
    
    print("⏳ pykrx 실전 펀더멘털 스냅샷(현재 및 1년 전) 수집 중...\n")
    
    # 4. Stage 4 스크리너 실행
    passed_stage4_df = screener.run(test_tickers_df, loader, base_date)
    
    print("📊 [입력된 테스트 종목 (ROE 포함)]")
    display(test_tickers_df)
    
    print("\n✅ [Stage 4 통과] 밸류에이션 검증 완료 종목")
    display(passed_stage4_df)
    
except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from datetime import date
from data.loader import QuantDataLoader
from stages.stage5_financial_health import FinancialHealthScreener

print("==================================================")
print("🚀 [실전 테스트] Stage 5: 재무 건전성 (Financial Health)")
print("==================================================\n")

# 1. 기준일 설정
base_date = date(2026, 7, 31)

# 2. 파라미터 설정
stage5_params = {
    'debt_ratio_percentile_cutoff': 0.5,  # 섹터 내 부채비율 하위 50%
    'interest_coverage_min': 1.5          # 이자보상배율 1.5배 이상
}

try:
    loader = QuantDataLoader()
    screener = FinancialHealthScreener(params=stage5_params)
    
    # 3. 테스트용 실제 종목 리스트 (Stage 4 통과 가정)
    # 금융주 예외 처리 검증을 위해 KB금융(105560) 포함
    test_tickers_df = pd.DataFrame({
        'ticker': ['005930', '000660', '005380', '105560'],  # 삼성전자, SK하이닉스, 현대차, KB금융
        'sector': ['IT', 'IT', '자동차', '금융']
    })
    
    print("⏳ DART API TTM(최근 4개 분기 합산) 재무 데이터 수집 중...\n")
    
    # 4. Stage 5 스크리너 실행
    passed_stage5_df = screener.run(test_tickers_df, loader, base_date)
    
    print("📊 [입력된 테스트 종목]")
    display(test_tickers_df)
    
    print("\n✅ [Stage 5 통과] 최종 재무 건전성 검증 완료 종목")
    display(passed_stage5_df)
    
except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from datetime import date
import yaml
import pickle
from data.loader import QuantDataLoader
from core.pipeline import QuantPipeline

print("==================================================")
print("🚀 [최종 통합 테스트] KOSPI 퀀트 파이프라인 전체 가동")
print("==================================================\n")

# 1. config/params.yaml 파일에서 최신 파라미터(Z-score 및 퍼센타일 설정 포함)를 동적으로 로드
try:
    with open('config/params.yaml', 'r', encoding='utf-8') as f:
        pipeline_params = yaml.safe_load(f)
    print("✅ config/params.yaml 설정 파일 로드 성공!")
except Exception as e:
    print(f"❌ config/params.yaml 파일을 읽는 중 에러 발생: {e}")
    raise e

# 2. 기준일 설정 (현재 날짜)
base_date = date(2026, 7, 31)

try:
    # 3. 로더 및 파이프라인 초기화
    loader = QuantDataLoader()
    pipeline = QuantPipeline(params=pipeline_params, loader=loader)
    
    print("⏳ 파이프라인 가동 중... (데이터 수집 및 분석에 시간이 소요됩니다.)\n")
    
    # 4. 파이프라인 실행 (최종 통과 종목 및 히스토리 반환)
    final_df, history = pipeline.run(base_date)
    
    # ---------------------------------------------------------
    # 5. 결과 및 단계별 탈락/통과 이력(History) 리포팅
    # ---------------------------------------------------------
    print("📈 [파이프라인 단계별 통과 현황]")
    
    if 'stage1' in history:
        print(f"✔️ Stage 1 (소외 섹터 선별) : {len(history['stage1'])}개 종목 통과")
    if 'stage2' in history:
        print(f"✔️ Stage 2 (섹터 내 우량주) : {len(history['stage2'])}개 종목 통과")
    if 'stage3' in history:
        print(f"✔️ Stage 3 (체질 개선)     : {len(history['stage3'])}개 종목 통과")
    if 'stage4' in history:
        print(f"✔️ Stage 4 (밸류에이션)   : {len(history['stage4'])}개 종목 통과")
    if 'stage5' in history:
        print(f"✔️ Stage 5 (재무 건전성)   : {len(history['stage5'])}개 종목 최종 생존")
        
    print("\n==================================================")
    print("🏆 [최종 산출된 조기 은퇴 포트폴리오 후보군]")
    print("==================================================")
    
    if not final_df.empty:
        display(final_df)
    else:
        print("조건을 모두 만족하는 종목이 이번 달에는 없습니다. (시장 상황에 따라 자연스러운 현상일 수 있습니다.)")
        
    print("\n💡 Tip: 특정 단계에서 어떤 종목들이 떨어졌는지 분석하고 싶다면,")
    print("        `history['stage3']` 처럼 딕셔너리를 호출하여 확인할 수 있습니다.")

except Exception as e:
    print(f"❌ 파이프라인 실행 중 에러 발생: {e}")

# 파이프라인 실행 완료 후 (final_df, history = pipeline.run(...) 이후)
# history 객체를 pkl 파일로 저장
with open('debug_history.pkl', 'wb') as f:
    pickle.dump(history, f)

print("✅ history 데이터가 'debug_history.pkl' 파일로 성공적으로 저장되었습니다.")